# WGCNA Module Analysis and GO Enrichment

Before running this notebook, please run the R codes (WGCNA_microglia(astrocyte).R). You will need pb_microglia(astrocyte)_control_filtered.csv to run the R codes. You will get three result csv files for each cell type. 
1. df_tom_similarity_microglia(astrocyte).csv
2. df_zsummary_microglia(astrocyte).csv
3. modules_whole_microglia(astrocyte).csv

Total 6 files should be copied on the ~/data/WGCNA_modules directory before running the code below. 

This notebook analyzes WGCNA modules for Microglia and Astrocytes, performs GO enrichment analysis using GSEApy, and visualizes the results.

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add sources directory to path
sys.path.append('../../sources')
import wgcna_utils as wu

In [2]:
# Parameters
root_path = "../../data/primary_cohort"
data_dir = f"{root_path}/WGCNA_modules"
output_base_dir = f"{root_path}/WGCNA_GO_enrichr"
z_summary_thres = 2
cell_types = ['microglia', 'astrocyte']

os.makedirs(output_base_dir, exist_ok=True)

In [3]:
# ── Global style ──────────────────────────────────
plt.rcParams.update({
    "font.family": "Liberation Serif",
    "font.size": 9,
    "mathtext.default": "regular",
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
})

for ct in cell_types:
    print(f"\n{'='*20} Processing: {ct} {'='*20}")
    
    # 1. Load modules filtered by Z-summary
    z_summary_path = os.path.join(data_dir, f'df_zsummary_{ct}.csv')
    module_csv_path = os.path.join(data_dir, f'modules_whole_{ct}.csv')
    
    ct_modules, background_genes, module_mapping = wu.get_module_dict(
        ct, z_summary_path, module_csv_path, z_summary_thres=z_summary_thres
    )
    
    # Save module mapping
    mapping_df = pd.DataFrame([module_mapping]).T
    mapping_df.columns = ['Module_Number']
    mapping_df.to_csv(os.path.join(data_dir, f'{ct}_module_number_mapping.csv'))
    
    # 2. Run GO Enrichment and Plot
    ct_output_dir = os.path.join(output_base_dir, f'GO_enrichr_{ct}')
    
    # Set plot color map based on cell type
    cmap = plt.cm.OrRd
    
    for m_idx in sorted(ct_modules.keys()):
        print(f"Processing {ct} Module {m_idx}...")
        
        # Run enrichment
        res_df = wu.run_go_enrichment(ct_modules[m_idx], ct, m_idx, ct_output_dir)
        
        # Plot results (top 10)
        if not res_df.empty:
            wu.plot_go_results(res_df, ct, m_idx, ct_output_dir, n_top=10, color_map=cmap)
            
    print(f"Done processing {ct}!")


==================== Processing: microglia ====================
Module 1: 369 genes
Module 2: 1586 genes
Module 3: 1257 genes
Module 4: 59 genes
Module 5: 66 genes
Module 6: 58 genes
Module 8: 424 genes
Module 9: 221 genes
Module 10: 104 genes
Module 11: 109 genes
Module 12: 92 genes
Module 13: 131 genes
Module 14: 326 genes
Module 15: 249 genes
Module 16: 403 genes
Module 17: 91 genes
Module 18: 154 genes
Module 19: 207 genes
Module 20: 4342 genes
Module 21: 630 genes
Processing microglia Module 1...
Processing microglia Module 2...
Processing microglia Module 3...
Processing microglia Module 4...
Processing microglia Module 5...
Processing microglia Module 6...
Processing microglia Module 8...
Processing microglia Module 9...
Processing microglia Module 10...
Processing microglia Module 11...
Processing microglia Module 12...
Processing microglia Module 13...
Processing microglia Module 14...
Processing microglia Module 15...
Processing microglia Module 16...
Processing microglia Mo

In [7]:
import os
import pandas as pd

for ct in ['astrocyte', 'microglia']:
    base_dir = "../../data/primary_cohort/WGCNA_modules"
    output_dir = os.path.join(base_dir, f"modules_{ct}")
    os.makedirs(output_dir, exist_ok=True)

    mapping_file = os.path.join(base_dir, f"{ct}_module_number_mapping.csv")
    modules_file = os.path.join(base_dir, f"modules_whole_{ct}.csv")

    df_map = pd.read_csv(mapping_file)
    color_col = df_map.columns[0]
    num_col = df_map.columns[1]
    color_to_num = dict(zip(df_map[color_col], df_map[num_col]))

    # exclude gold
    if 'gold' in color_to_num:
        del color_to_num['gold']

    df_genes = pd.read_csv(modules_file)
    df_filtered = df_genes[df_genes['dynamicColors'].isin(color_to_num.keys())].copy()
    df_filtered['module_id'] = df_filtered['dynamicColors'].map(color_to_num).astype(int)

    # {ct}_all_modules.tsv 
    df_tsv = df_filtered[['module_id', 'genes']].sort_values(by=['module_id', 'genes'])
    tsv_path = os.path.join(output_dir, f"{ct}_all_modules.tsv")
    df_tsv.to_csv(tsv_path, sep='\t', index=False)
    grouped = df_tsv.groupby('module_id')

    for mod_id, group in grouped:
       
        file_name = f"{ct}_module_{mod_id:02d}.txt"
        file_path = os.path.join(output_dir, file_name)
        
        gene_list = group['genes'].tolist()
        
        with open(file_path, 'w') as f:
            for gene in gene_list:
                f.write(f"{gene}\n")

    print("Module file generated!")
    print(f"output path: {output_dir}")


Module file generated!
output path: ../../data/primary_cohort/WGCNA_modules/modules_astrocyte
Module file generated!
output path: ../../data/primary_cohort/WGCNA_modules/modules_microglia
